# Hash Tables, Equality, and Collisions


```{contents}
:local:
:depth: 2
```


```{index} A Collision Is Not Evidence That Keys Are Equal
```



```{index} A Collision Is Not Evidence That Keys Are Equal
```

## A Collision Is Not Evidence That Keys Are Equal



A hash function maps keys to a finite bucket range, so different keys can map to the same bucket. The hash narrows the search; equality identifies the actual key. In a table of five buckets using a simple remainder demonstration, integer keys 2, 7 and 12 all map to bucket two. Keeping only one value at that position would lose entries unless the ADT intentionally replaces an equal key.

A chained bucket can examine the entries for actual equality. An open-addressed representation follows a probe sequence instead. Removing an entry in open addressing must preserve the ability to find later entries whose probe passed through its slot. Turning a deleted slot directly into never-used can stop a search too soon; the distinction between deleted and never occupied matters.

Guided trace: insert the three colliding keys, remove seven, then find twelve. Explain why a search must still reach twelve. Now grow the bucket range and recompute placement; copying each entry to its old numeric slot is not sufficient because the mapping depends on the table size. Hash distribution, equality consistency, load and deletion policy work together.

For related explanation, see the further reading below.[^development-reading]


```{index} Dictionary Contracts and Key Equality
```

## Dictionary Contracts and Key Equality




```{index} dictionary; key equality; hashing
```

A dictionary associates one value with each distinct key. Looking up a customer ID asks for that ID's record; it does not ask for the first record in order. C# `Dictionary<TKey, TValue>` uses hashing for key-based access. `Add` rejects an existing key, indexer assignment inserts or replaces, and `TryGetValue` reports whether a key exists without treating a missing key as a successful default value.

In [1]:
using System;
using System.Collections.Generic;
using System.Linq;

var owners = new Dictionary<string, string>(StringComparer.OrdinalIgnoreCase);
owners.Add("T-100", "Amina");
owners["t-100"] = "Diego";
Console.WriteLine($"count={owners.Count}, owner={owners["T-100"]}");
Console.WriteLine($"missing={owners.TryGetValue("T-999", out _)}");
Console.WriteLine($"removed={owners.Remove("T-100")}, count={owners.Count}");

count=1, owner=Diego
missing=False
removed=True, count=0


This dictionary treats `T-100` and `t-100` as equal, so assignment changes the owner rather than adding another key. Choose the equality policy when creating the collection and apply it consistently. For symbolic ticket IDs, ordinal comparison avoids culture-dependent language rules. This example deliberately ignores case; other applications may need case-sensitive identifiers.

Hashing and equality must agree: equal keys must have equal hash codes. Different keys may have the same hash code, so a lookup must still compare actual keys. A custom comparer supplies both operations. Do not modify fields that determine a stored key's equality or hash code; its new bucket may no longer match the location used during insertion. Strings are immutable, which avoids that particular mutation hazard. [Microsoft's GetHashCode contract](https://learn.microsoft.com/en-us/dotnet/api/system.object.gethashcode) explains this agreement.

```{index} Hash Codes and Bucket Mapping
```

## Hash Codes and Bucket Mapping




```{index} hash function; bucket; negative hash code
```

Separate two steps: compute an integer hash code, then map that code into one of m buckets. Hash codes can be negative. C# remainder can also be negative, and `Math.Abs(int.MinValue)` cannot fit in an `int`. Reinterpret the code as an unsigned integer before remainder to obtain a valid index without that overflow.

In [1]:
using System;
int[] codes = { 12, -1, int.MinValue, int.MaxValue };
int bucketCount = 5;
foreach (int hash in codes)
{
    int bucket = (int)(unchecked((uint)hash) % (uint)bucketCount);
    Console.WriteLine($"hash={hash}, bucket={bucket}");
}

hash=12, bucket=2
hash=-1, bucket=0
hash=-2147483648, bucket=3
hash=2147483647, bucket=2


`unchecked((uint)hash)` keeps the original bits and interprets them as a nonnegative number; it does not claim that the code is a unique identifier. BucketCount must be positive. The result lies from zero through m - 1. Equal hash codes always map to the same bucket for a fixed m, but many different codes also map there.

String hash codes may vary across processes or platforms. Use them for temporary placement, not as permanent customer IDs or persisted checksums. The teaching map below uses integer IDs themselves as simple hash codes so its collision trace is reproducible. This is an illustration, not a general recommendation for distributing arbitrary business keys.

**Common mistake.** Writing `Math.Abs(hash) % bucketCount`. For `int.MinValue` the absolute value overflows and throws, and for other negative codes it silently maps to a different bucket than the unsigned reinterpretation; the unchecked cast is the version that works for every code.


```{index} What Makes a Hash Function Good
```

### What Makes a Hash Function Good

Everything a hash table promises depends on the hash function spreading keys evenly across the buckets, and real keys are not random. A good hash function has three properties. It is cheap, because it runs on every lookup. It is deterministic, so that a key always maps to the same bucket while the table size is fixed. And it depends on *all* of the key, so that keys which differ in any part land in different buckets more often than not.

The failures are predictable. Using the remainder of an integer key by a power of two looks only at the key's low bits, so keys that are all even, or all multiples of 100, crowd into a fraction of the buckets. Taking the first character of a string as the bucket, a form of *binning*, clusters names under the common initials. Keys that share a structure, such as product codes that all begin with the same department prefix, defeat any function that reads only the prefix. The usual remedies are to use a prime bucket count, so the remainder mixes all the key's digits, and for strings to *fold* every character into the result with multiplication, so that position and value both matter. The example hashes the same product codes three ways and reports how unevenly each spreads them over 11 buckets.

In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

string[] codes = { "HW-1000", "HW-1010", "HW-1020", "HW-1030", "HW-1040", "SW-1000", "SW-1005", "SW-1010", "SW-1015", "SW-1020", "SV-2000" };
int m = 11;

Report("first character", codes, c => c[0] % m);
Report("sum of characters", codes, c => c.Sum(ch => (int)ch) % m);
Report("fold with multiplier", codes, c => { int h = 0; foreach (char ch in c) h = unchecked(h * 31 + ch); return (int)(unchecked((uint)h) % (uint)m); });

static void Report(string label, string[] keys, Func<string, int> hash)
{
    int[] chainLengths = new int[11];
    foreach (string key in keys) chainLengths[hash(key)]++;
    int used = chainLengths.Count(len => len > 0);
    Console.WriteLine($"{label,-21} buckets used={used,2}  longest chain={chainLengths.Max()}  layout=[{string.Join(" ", chainLengths)}]");
}

first character       buckets used= 1  longest chain=11  layout=[0 0 0 0 0 0 11 0 0 0 0]
sum of characters     buckets used= 7  longest chain=3  layout=[0 3 2 2 1 1 1 1 0 0 0]
fold with multiplier  buckets used= 9  longest chain=2  layout=[1 1 1 1 1 0 2 0 2 1 1]


The first-character hash puts all eleven keys into one bucket. Only three different letters begin the codes, and by bad luck the character codes of `H` and `S`, 72 and 83, leave the same remainder modulo 11, so every lookup compares against the whole table; a hash that reads part of the key inherits every regularity in that part. Summing the characters uses seven buckets with a longest chain of three, but it still collides whenever two codes are rearrangements of each other or differ by offsetting amounts, as `HW-1010` and `HW-1100` would. The folding hash spreads the keys over nine of the eleven buckets with a longest chain of two, because multiplying by 31 before adding each character makes position matter as well as value. None of these is a cryptographic function and none is unique; the goal is a flat layout, which the library's string hash achieves with a stronger version of the fold. For your own key types, the practical rule is to combine every field that participates in equality, which `HashCode.Combine` does, and never to hash on a subset of them.

```{index} Separate Chaining and Complete Operations
```

## Separate Chaining and Complete Operations




```{index} separate chaining; collision; hash-table update
```

A collision occurs when different keys choose the same bucket. **Separate chaining** stores a collection of entries at each bucket. A lookup computes the bucket and compares keys within that chain. An update must search the chain before appending; otherwise it can create duplicate keys. Removal must preserve the other colliding entries.

With four buckets, IDs 101, 105, and 109 all choose bucket one. To find 109, inspect 101, then 105, then 109. To report that 113 is absent, inspect the entire chain. A successful lookup may stop early; an unsuccessful one must establish that no matching key exists.

The complete `TicketMap` stores integer ticket IDs and string owners. `Put` returns true for a new key and false for replacement. `Remove` reports whether membership changed. `TryGetValue` returns false on absence; ignore its output value in that case.

In [1]:
using System;
using System.Collections.Generic;
using System.Linq;

var owners = new TicketMap();
owners.Put(101, "Amina"); owners.Put(105, "Diego"); owners.Put(109, "Mina");
Console.WriteLine("before=" + owners.DescribeBuckets());
Console.WriteLine($"replacement inserted={owners.Put(105, "Lee")}, count={owners.Count}");
owners.Put(113, "Noor");
Console.WriteLine($"after growth: buckets={owners.BucketCount}, count={owners.Count}");
Console.WriteLine(owners.DescribeBuckets());
owners.Remove(101);
Console.WriteLine($"109 survives={owners.TryGetValue(109, out string owner)}, owner={owner}");
Console.WriteLine($"missing={owners.TryGetValue(101, out _)}, count={owners.Count}");

public class TicketMap
{
    private List<Entry>[] buckets = CreateBuckets(4);
    public int Count { get; private set; }
    public int BucketCount => buckets.Length;
    public double LoadFactor => (double)Count / buckets.Length;

    public bool Put(int id, string owner)
    {
        int index = Bucket(id, buckets.Length);
        for (int i = 0; i < buckets[index].Count; i++)
        {
            if (buckets[index][i].Id == id)
            {
                buckets[index][i] = new Entry(id, owner);
                return false;
            }
        }
        if ((Count + 1.0) / buckets.Length > 0.75)
        {
            Grow();
            index = Bucket(id, buckets.Length);
        }
        buckets[index].Add(new Entry(id, owner));
        Count++;
        return true;
    }

    public bool TryGetValue(int id, out string owner)
    {
        foreach (Entry entry in buckets[Bucket(id, buckets.Length)])
        {
            if (entry.Id == id) { owner = entry.Owner; return true; }
        }
        owner = default!;
        return false;
    }

    public bool Remove(int id)
    {
        List<Entry> chain = buckets[Bucket(id, buckets.Length)];
        for (int i = 0; i < chain.Count; i++)
        {
            if (chain[i].Id != id) continue;
            chain.RemoveAt(i);
            Count--;
            return true;
        }
        return false;
    }

    private void Grow()
    {
        List<Entry>[] replacement = CreateBuckets(buckets.Length * 2);
        foreach (List<Entry> chain in buckets)
            foreach (Entry entry in chain)
                replacement[Bucket(entry.Id, replacement.Length)].Add(entry);
        buckets = replacement;
    }

    private static int Bucket(int id, int length) => (int)(unchecked((uint)id) % (uint)length);

    private static List<Entry>[] CreateBuckets(int length)
    {
        var result = new List<Entry>[length];
        for (int i = 0; i < length; i++) result[i] = new List<Entry>();
        return result;
    }

    public string DescribeBuckets()
    {
        var lines = new List<string>();
        for (int i = 0; i < buckets.Length; i++)
            lines.Add($"{i}: {string.Join(", ", buckets[i].Select(e => e.Id))}");
        return string.Join(" | ", lines);
    }

    private class Entry
    {
        public int Id { get; }
        public string Owner { get; }
        public Entry(int id, string owner) { Id = id; Owner = owner; }
    }
}

before=0:  | 1: 101, 105, 109 | 2:  | 3: 
replacement inserted=False, count=3
after growth: buckets=8, count=4
0:  | 1: 105, 113 | 2:  | 3:  | 4:  | 5: 101, 109 | 6:  | 7: 
109 survives=True, owner=Mina
missing=False, count=3


Each stored entry belongs to the bucket computed from its ID and the current bucket count. Count equals the total number of entries, and each ID occurs once. The internal entries are immutable; replacement installs a new entry with the same ID. An update returns before increasing Count or testing growth because it adds no key.

After removal, another colliding ID remains reachable in the same chain. Our chains are lists, so removal may shift later entries within that chain. The cost depends on chain length rather than the total number of buckets. This map keeps capacity after deletions and has no iterator API; production libraries provide more policies and interfaces.

**Visualize it.** OpenDSA's [Open Hashing](https://opendsa-server.cs.vt.edu/OpenDSA/Books/CS3/html/OpenHash.html) exercise animates separate chaining; note that OpenDSA calls chaining *open hashing* and calls open addressing *closed hashing*.

**Common mistake.** Appending to the chain without scanning it first. The same ID is then stored twice, `Count` overstates the entries, and a later `Remove` deletes only the first copy while the stale one keeps answering lookups.


```{index} Load Factor and Rehashing
```

## Load Factor and Rehashing




```{index} load factor; resizing; rehashing
```

Define load factor α = n / m, where n is the number of entries and m is the number of buckets. Before adding 113, the map has three entries in four buckets, so α = 0.75. A fourth new key would exceed the teaching threshold, so the map doubles to eight buckets before inserting it.

Changing the bucket count changes placement. Copying old bucket lists to the same array indexes is insufficient. Recompute every entry's bucket under the new capacity, a process called **rehashing**.

| ID | Bucket with m = 4 | Bucket with m = 8 |
| --- | ---: | ---: |
| 101 | 1 | 5 |
| 105 | 1 | 1 |
| 109 | 1 | 5 |
| 113 | 1 | 1 |

Growth splits this collision group into two chains. It does not guarantee that all collisions disappear. A poor hash function can keep many keys in one chain even at low load factor.

Under well-distributed hashes and constant-time hashing/equality, chaining operations have expected O(1 + α) work. A pathological chain can require O(n) work. This resize visits all entries and initializes a larger bucket array, giving O(n + m) work for that operation. Geometric growth contributes amortized constant work per new key when load is kept bounded; one growth-triggering insertion can still be linear.

The threshold 0.75 belongs to this teaching map, not to a universal hash-table rule. Separate chaining permits α greater than one. Open addressing needs unused slots and becomes much more expensive as occupancy approaches one. Key hashing itself is an additional cost, such as scanning characters in a long string.

```{index} Expected Cost as a Function of Load
```

### Expected Cost as a Function of Load

Under the assumption that keys spread evenly, the cost of a hash table operation depends on α alone, not on n. For separate chaining, an unsuccessful search examines a whole chain of expected length α, and a successful search examines about half of one, 1 + α / 2 comparisons. For open addressing with linear probing the formulas are steeper, because a probe that leaves its home slot competes with neighboring clusters: roughly (1 + 1 / (1 - α)) / 2 probes for a successful search and (1 + 1 / (1 - α)²) / 2 for an unsuccessful one. The example evaluates both sets of formulas at several loads.

In [ ]:
using System;

Console.WriteLine("  load   chaining hit  chaining miss   probing hit  probing miss");
foreach (double a in new[] { 0.25, 0.5, 0.75, 0.9, 0.95 })
{
    double chainHit = 1 + a / 2, chainMiss = a;
    double probeHit = 0.5 * (1 + 1 / (1 - a)), probeMiss = 0.5 * (1 + 1 / ((1 - a) * (1 - a)));
    Console.WriteLine($"  {a,4:F2}   {chainHit,12:F2}  {chainMiss,13:F2}   {probeHit,11:F2}  {probeMiss,12:F2}");
}

  load   chaining hit  chaining miss   probing hit  probing miss
  0.25           1.12           0.25          1.17          1.39
  0.50           1.25           0.50          1.50          2.50
  0.75           1.38           0.75          2.50          8.50
  0.90           1.45           0.90          5.50         50.50
  0.95           1.48           0.95         10.50        200.50


At half load both strategies cost one or two comparisons per operation, which is the "expected O(1)" that makes hash tables attractive. The difference appears as the table fills: at 90 percent load an unsuccessful linear-probing search expects about 50 probes while chaining still expects fewer than one chain entry per miss. That is why open-addressed tables resize at a lower threshold than chained ones, commonly around 0.5 to 0.7, and why chaining tolerates α above 1 at all. The formulas describe averages under an even spread; a poor hash function violates the assumption, and then no load factor rescues the table.

| Probe sequence | Rule for the i-th probe after the home slot | What it fixes |
| --- | --- | --- |
| Linear probing | home + i | Nothing; simple, but runs of occupied slots grow into each other (primary clustering) |
| Linear probing by steps | home + c × i, with c and the table size sharing no factor | Spreads neighbors apart but keys with the same home still follow one path |
| Quadratic probing | home + i² | Breaks up runs; must guarantee the sequence visits enough slots |
| Double hashing | home + i × h2(key) | Different keys with the same home take different paths; the best spread |

Each row changes only the probe sequence; the tombstone rule and the equality check from the open-addressing example stay the same.

```{index} Open Addressing and Deleted Slots
```

## Open Addressing and Deleted Slots




```{index} open addressing; linear probing; tombstone
```

**Open addressing** stores entries directly in the table. **Linear probing** tries the chosen slot, then subsequent slots with wraparound until it finds the key or a place to insert. A lookup may stop at a slot that has never been occupied, because insertion would have stopped there too. It cannot stop merely because a previous entry was removed.

For capacity seven, IDs 10, 17, and 24 all start at slot three. Linear probing stores them at three, four, and five. Clearing slot three after deleting 10 would make a naive search for 17 stop early. A **tombstone** marks a deleted slot through which lookup must continue.

In [1]:
using System;
int?[] slots = new int?[7];
slots[3] = 10; slots[4] = 17; slots[5] = 24;
bool[] deleted = new bool[7];
slots[3] = null;
deleted[3] = true;
Console.WriteLine($"naive finds 17={Find(17, false)}");
Console.WriteLine($"tombstone finds 17={Find(17, true)}");
Console.WriteLine($"missing 31={Find(31, true)}");

bool Find(int key, bool honorTombstones)
{
    int index = (int)(unchecked((uint)key) % (uint)slots.Length);
    for (int probes = 0; probes < slots.Length; probes++)
    {
        if (slots[index] == key) return true;
        if (slots[index] is null && (!honorTombstones || !deleted[index])) return false;
        index = (index + 1) % slots.Length;
    }
    return false;
}

naive finds 17=False
tombstone finds 17=True
missing 31=False


Limit a search to at most m probes so a full table cannot cause an endless loop. Insertion can reuse a tombstone, but it must still search far enough to rule out an existing equal key. Otherwise it can add a duplicate before reaching the old copy. Over time, many tombstones lengthen searches; rebuilding the table can remove them.

This example demonstrates a deletion rule, not a complete open-addressed dictionary. The chapter's complete implementation uses chaining. Both strategies must combine placement with actual key equality; neither treats a collision as a duplicate by itself.

**Visualize it.** OpenDSA's [Collision Resolution](https://opendsa-server.cs.vt.edu/OpenDSA/Books/CS3/html/HashCSimple.html) slideshows animate linear probing and its clustering, and [Hashing Deletion](https://opendsa-server.cs.vt.edu/OpenDSA/Books/CS3/html/HashDel.html) shows why a removed slot needs a tombstone.

**Common mistake.** Clearing a deleted slot to `null`. Any key whose probe sequence passed through that slot becomes unreachable even though it is still stored; the tombstone exists so that lookups keep probing past a deletion.


The complete teaching map uses separate chaining. The probing/tombstone example is an optional comparison, not another full map implementation you must build. The next section applies dictionary and set contracts to business reconciliation.


```{index} Code Tracing: Linear Probing Placement
```

### Code Tracing: Linear Probing Placement

A table of 7 slots uses the home slot `key % 7` and linear probing. Insert the keys 15, 22, 8, 30, 9, 16 in that order. For each key write its home slot, the slots it probes, and the slot where it lands. Then say how many probes an unsuccessful search for 23 would make before stopping at an empty slot. The solution prints the trace.

In [ ]:
// Solution
using System;
using System.Collections.Generic;

int?[] slots = new int?[7];
foreach (int key in new[] { 15, 22, 8, 30, 9, 16 })
{
    int home = key % 7, index = home;
    var probed = new List<int>();
    while (slots[index] is not null) { probed.Add(index); index = (index + 1) % 7; }
    slots[index] = key;
    Console.WriteLine($"key {key,2}: home {home}, occupied {(probed.Count == 0 ? "none" : string.Join(",", probed)),-7} -> slot {index}");
}
Console.WriteLine($"table: [{string.Join(" ", Array.ConvertAll(slots, s => s is null ? "-" : s.ToString()))}]");

int probes = 0, i = 23 % 7;
while (slots[i] is not null && slots[i] != 23) { probes++; i = (i + 1) % 7; }
Console.WriteLine($"search 23: home {23 % 7}, {probes} occupied slots probed before the empty slot {i}");

key 15: home 1, occupied none    -> slot 1
key 22: home 1, occupied 1       -> slot 2
key  8: home 1, occupied 1,2     -> slot 3
key 30: home 2, occupied 2,3     -> slot 4
key  9: home 2, occupied 2,3,4   -> slot 5
key 16: home 2, occupied 2,3,4,5 -> slot 6
table: [- 15 22 8 30 9 16]
search 23: home 2, 5 occupied slots probed before the empty slot 0


The keys 15, 22, and 8 all have home slot 1 and form a run in slots 1, 2, 3. Key 30 has home slot 2, finds it occupied, and lands in slot 4; key 9 probes slots 2, 3, 4 and lands in slot 5; key 16 probes four occupied slots and lands in slot 6. The run has grown from three slots to six even though only three keys shared a home: that is primary clustering, and it is why the search for 23, whose home slot 2 is inside the run, probes five occupied slots before reaching the empty slot 0 and concluding that 23 is absent. Under separate chaining the same search would examine only bucket 2's chain, which holds 30, 9, and 16, three comparisons.

```{index} Check Your Understanding
```

### Check Your Understanding

Answer each question before opening its dropdown.

1. Two different customer records produce the same hash code. Is that a defect in the hash function, in `Equals`, or in neither?

```{dropdown} Answer
Neither, by itself. A hash function maps a large key space into a small range, so collisions are inevitable; the contract only requires that equal keys produce equal codes. The table resolves the collision by comparing the actual keys. It becomes a performance problem only when many keys collide systematically.
```

2. A table has 1,000 buckets and 1,500 entries under separate chaining. Is it broken, and about how many comparisons does a successful lookup expect?

```{dropdown} Answer
Not broken: chaining allows a load factor above 1. With α = 1.5 and an even spread, a successful lookup expects about 1 + 1.5 / 2, under two comparisons. The same load would be impossible for open addressing, which cannot store more entries than slots.
```

3. Why does doubling the bucket array require every entry to be rehashed rather than copied to the same index?

```{dropdown} Answer
The bucket is the remainder of the hash code by the bucket count, so changing the count changes almost every key's bucket. An entry copied to its old index would be in the wrong bucket under the new count and would never be found again.
```

4. A string key is stored in a dictionary and then the program needs to change the key's spelling. What is the only safe sequence?

```{dropdown} Answer
Remove the entry under the old key, then add it under the new key. Strings are immutable so this hazard cannot arise by mutation, but for a custom key type the same rule applies: never change a field that participates in equality or hashing while the object is stored.
```

```{index} Practice: Find a Colliding Key
```

## Practice: Find a Colliding Key




```{index} Practice: Find a Colliding Key
```


Complete the operation, run the cell, and explain why the chosen item is returned.


In [ ]:
using System;
using System.Collections.Generic;

TicketMap map = new TicketMap();
map.Put(1, "First"); map.Put(9, "Second");
// TODO: explain the result.
Console.WriteLine(map.TryGetValue(9, out string value) ? value : "missing");

public class TicketMap
{
    private List<Entry>[] buckets = CreateBuckets(4);
    public int Count { get; private set; }
    public int BucketCount => buckets.Length;
    public double LoadFactor => (double)Count / buckets.Length;

    public bool Put(int id, string owner)
    {
        int index = Bucket(id, buckets.Length);
        for (int i = 0; i < buckets[index].Count; i++)
        {
            if (buckets[index][i].Id == id)
            {
                buckets[index][i] = new Entry(id, owner);
                return false;
            }
        }
        if ((Count + 1.0) / buckets.Length > 0.75)
        {
            Grow();
            index = Bucket(id, buckets.Length);
        }
        buckets[index].Add(new Entry(id, owner));
        Count++;
        return true;
    }

    public bool TryGetValue(int id, out string owner)
    {
        foreach (Entry entry in buckets[Bucket(id, buckets.Length)])
        {
            if (entry.Id == id) { owner = entry.Owner; return true; }
        }
        owner = default!;
        return false;
    }

    public bool Remove(int id)
    {
        List<Entry> chain = buckets[Bucket(id, buckets.Length)];
        for (int i = 0; i < chain.Count; i++)
        {
            if (chain[i].Id != id) continue;
            chain.RemoveAt(i);
            Count--;
            return true;
        }
        return false;
    }

    private void Grow()
    {
        List<Entry>[] replacement = CreateBuckets(buckets.Length * 2);
        foreach (List<Entry> chain in buckets)
            foreach (Entry entry in chain)
                replacement[Bucket(entry.Id, replacement.Length)].Add(entry);
        buckets = replacement;
    }

    private static int Bucket(int id, int length) => (int)(unchecked((uint)id) % (uint)length);

    private static List<Entry>[] CreateBuckets(int length)
    {
        var result = new List<Entry>[length];
        for (int i = 0; i < length; i++) result[i] = new List<Entry>();
        return result;
    }

    public string DescribeBuckets()
    {
        var lines = new List<string>();
        for (int i = 0; i < buckets.Length; i++)
        {
            var ids = new List<int>();
            foreach (Entry entry in buckets[i]) ids.Add(entry.Id);
            lines.Add($"{i}: {string.Join(", ", ids)}");
        }
        return string.Join(" | ", lines);
    }

    private class Entry
    {
        public int Id { get; }
        public string Owner { get; }
        public Entry(int id, string owner) { Id = id; Owner = owner; }
    }
}


In [1]:
using System;
using System.Collections.Generic;

TicketMap map = new TicketMap();
map.Put(1, "First"); map.Put(9, "Second");
Console.WriteLine(map.TryGetValue(9, out string value) ? value : "missing");

public class TicketMap
{
    private List<Entry>[] buckets = CreateBuckets(4);
    public int Count { get; private set; }
    public int BucketCount => buckets.Length;
    public double LoadFactor => (double)Count / buckets.Length;

    public bool Put(int id, string owner)
    {
        int index = Bucket(id, buckets.Length);
        for (int i = 0; i < buckets[index].Count; i++)
        {
            if (buckets[index][i].Id == id)
            {
                buckets[index][i] = new Entry(id, owner);
                return false;
            }
        }
        if ((Count + 1.0) / buckets.Length > 0.75)
        {
            Grow();
            index = Bucket(id, buckets.Length);
        }
        buckets[index].Add(new Entry(id, owner));
        Count++;
        return true;
    }

    public bool TryGetValue(int id, out string owner)
    {
        foreach (Entry entry in buckets[Bucket(id, buckets.Length)])
        {
            if (entry.Id == id) { owner = entry.Owner; return true; }
        }
        owner = default!;
        return false;
    }

    public bool Remove(int id)
    {
        List<Entry> chain = buckets[Bucket(id, buckets.Length)];
        for (int i = 0; i < chain.Count; i++)
        {
            if (chain[i].Id != id) continue;
            chain.RemoveAt(i);
            Count--;
            return true;
        }
        return false;
    }

    private void Grow()
    {
        List<Entry>[] replacement = CreateBuckets(buckets.Length * 2);
        foreach (List<Entry> chain in buckets)
            foreach (Entry entry in chain)
                replacement[Bucket(entry.Id, replacement.Length)].Add(entry);
        buckets = replacement;
    }

    private static int Bucket(int id, int length) => (int)(unchecked((uint)id) % (uint)length);

    private static List<Entry>[] CreateBuckets(int length)
    {
        var result = new List<Entry>[length];
        for (int i = 0; i < length; i++) result[i] = new List<Entry>();
        return result;
    }

    public string DescribeBuckets()
    {
        var lines = new List<string>();
        for (int i = 0; i < buckets.Length; i++)
        {
            var ids = new List<int>();
            foreach (Entry entry in buckets[i]) ids.Add(entry.Id);
            lines.Add($"{i}: {string.Join(", ", ids)}");
        }
        return string.Join(" | ", lines);
    }

    private class Entry
    {
        public int Id { get; }
        public string Owner { get; }
        public Entry(int id, string owner) { Id = id; Owner = owner; }
    }
}


Second


The output is `Second`. With four buckets, IDs 1 and 9 both map to bucket 1, because 9 % 4 is 1, so they share a chain. The lookup computed that bucket, walked the chain comparing actual IDs, passed entry 1, and matched entry 9; a collision slowed the lookup by one comparison but did not confuse the two keys.


```{rubric} Footnotes
```


[^1901_hashing-1]: Equal keys require equal hash codes; equal hash codes do not imply equal keys. Bucket positions and string hash codes are not persistent business identifiers.


[^development-reading]: Further reading: [OpenDSA CS3, Introduction to Hashing](https://opendsa-server.cs.vt.edu/OpenDSA/Books/CS3/html/HashIntro.html). The worked business scenarios and guided reasoning here are original teaching examples.
